# 1 - Imports

In [5]:
%reload_ext autoreload
%autoreload 2

In [6]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0].parents[0]))

In [7]:
from src.utils import config, io
from src.preprocessing import preprocess_pipeline
from src.models import model_pipeline, evaluate

# 2 - Preprocessing

In [8]:
split_cfg = io.load_json(config.PROCESSED_DATA_DIR / 'splits/temporal_v1.json')
split_cfg

{'description': 'Forecasting split for future country risk prediction',
 'train_years': [1999, 2017],
 'val_years': [2018, 2020],
 'test_years': [2021, 2024]}

In [9]:
def temporal_split(X, y, config):
    train_mask = (
        (X['YEAR'] >= config['train_years'][0]) &
        (X['YEAR'] <= config['train_years'][1])
    )

    val_mask = (
        (X['YEAR'] >= config['val_years'][0]) &
        (X['YEAR'] <= config['val_years'][1])
    )

    test_mask = (
        (X['YEAR'] >= config['test_years'][0]) &
        (X['YEAR'] <= config['test_years'][1])
    )

    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val     = X[val_mask], y[val_mask]
    X_test, y_test   = X[test_mask], y[test_mask]

    return X_train, y_train, X_val, y_val, X_test, y_test


In [10]:
import numpy as np
import pandas as pd

def expanding_window_split(df, start_year, end_year, val_start_year):

    for val_year in range(val_start_year, end_year + 1):
        
        train_mask = (df['YEAR'] >= start_year) & (df['YEAR'] < val_year)
        val_mask = df['YEAR'] == val_year
        
        train_idx = df.index[train_mask]
        val_idx = df.index[val_mask]
        
        if len(val_idx) > 0:
            yield train_idx, val_idx


# 3 - Parameter Tuning Logistic Regression Model

In [11]:
from src.features import selection, pruning

In [19]:
def get_dataset_feature_selected(dataset, params):
    non_float_features_t = list(dataset.columns[dataset.dtypes!=float])

    print('Initial Dataset Shape:', dataset.shape)
    dataset = dataset[dataset['OECD_RATING'] != '-']
    print('Drop Null Target Shape:', dataset.shape)
    dataset = selection.filter_missingness(
        dataset, 
        max_missing_ratio=params['max_missing_ratio']
    )
    print('Filter Missingness Shape:', dataset.shape)
    dataset = selection.filter_low_variance(
        dataset, 
        threshold=params['low_var_threshold']
    )
    print('Filter Low Variance Shape:', dataset.shape)
    dataset = selection.filter_correlated(
        dataset,
        max_corr=params['max_corr']
    )
    print('Filter Correlated Shape:', dataset.shape)
    # dataset = selection.select_by_mutual_information(
    #     dataset.drop(columns=['OECD_RATING']),
    #     dataset['OECD_RATING'],
    #     top_k=params['top_k_mi']
    # )
    # print('Filter Mutual Information Shape:', dataset.shape)
    print('Final Dataset Shape:', dataset.shape)
    return dataset

def get_X_y(dataset):
    # Drop Instances with little data
    dataset = dataset[(dataset.isna().sum(axis=1) / len(dataset.columns)) <= 0.5]

    non_float_features_t = list(dataset.columns[dataset.dtypes!=float])
    non_float_features = []
    for c in non_float_features_t:
        if c != 'OECD_RATING':
            non_float_features.append(c)

    X = dataset.drop(columns=['OECD_RATING']) # Drop Target
    X = X.drop(columns=['ISO3_COUNTRY_CODE']) # Drop Country ID (maybe YEAR)
    y = dataset['OECD_RATING']

    return X, y

def transform_initial_dataset(dataset, feature_selection_params):

    dataset = get_dataset_feature_selected(dataset, feature_selection_params)
    X, y = get_X_y(dataset)

    return X, y

## 3.1 - Parameter Tuning Feature Selection

In [13]:
dataset = io.load_csv(config.INTERIM_DATA_DIR / 'merged_dataset.csv', index_col=0)
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.MKTP.PP.CD,...,SE.PRM.CUAT.ZS,SP.POP.DPND,SL.TLF.CACT.ZS,GE.EST,RQ.EST,IC.BRE.BI.OS,IC.BRE.BE.OS,IQ.CPA.PADM.XQ,FS.AST.PRVT.GD.ZS,FM.AST.PRVT.GD.ZS
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-1999,1999,AFG,7,MEA,MNA,IDX,LIC,NaN,NaN,NaN,...,NaN,108.686031,46.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG-2000,2000,AFG,7,MEA,MNA,IDX,LIC,3.521418e+09,NaN,1.637703e+10,...,NaN,109.586048,46.562,-2.173946,-2.080253,NaN,NaN,NaN,NaN,NaN
AFG-2001,2001,AFG,-,MEA,MNA,IDX,LIC,2.813572e+09,-9.431974,1.516633e+10,...,NaN,110.219341,46.526,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG-2002,2002,AFG,-,MEA,MNA,IDX,LIC,3.825701e+09,28.600001,1.980700e+10,...,NaN,110.534611,46.505,-1.587687,-1.811546,NaN,NaN,NaN,NaN,NaN
AFG-2003,2003,AFG,-,MEA,MNA,IDX,LIC,4.520947e+09,8.832278,2.198200e+10,...,NaN,110.557540,46.497,-1.175768,-1.463108,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2019,2019,ZWE,7,SSF,SSA,IDB,LMC,3.335770e+10,-6.332450,6.361373e+10,...,75.001228,85.447906,65.795,-1.310435,-1.486515,NaN,NaN,3.0,3.428022,3.428022
ZWE-2020,2020,ZWE,7,SSF,SSA,IDB,LMC,3.198033e+10,-7.816951,6.488043e+10,...,NaN,84.384381,64.665,-1.342368,-1.434415,NaN,NaN,3.0,3.642132,3.642132
ZWE-2021,2021,ZWE,7,SSF,SSA,IDB,LMC,4.128767e+10,8.468017,7.625453e+10,...,NaN,83.384953,65.397,-1.290561,-1.386109,NaN,NaN,3.0,4.759522,4.759522


In [14]:
feature_selection_grid = {
    'max_missing_ratio': [0.2, 0.3, 0.4, 0.5, 0.6],
    'low_var_threshold': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'max_corr': [0.7, 0.75, 0.8, 0.85, 0.9, 0.95],
    'top_k_mi': [25, 50, 100, 200]
}

In [42]:
from itertools import product
from sklearn.metrics import roc_auc_score
import mlflow
import numpy as np

for max_missing_ratio, low_var_threshold, max_corr, top_k_mi in product(
    feature_selection_grid['max_missing_ratio'], feature_selection_grid['low_var_threshold'], feature_selection_grid['max_corr'], feature_selection_grid['top_k_mi'], 
):

    X, y = transform_initial_dataset(
        dataset, 
        {
            'max_missing_ratio': max_missing_ratio, 'low_var_threshold': low_var_threshold, 'max_corr': max_corr, 'top_k_mi': top_k_mi
        }
    )

    cv_scores = []
    with mlflow.start_run(run_name='logreg_expanding_cv'):

        for fold, (train_idx, val_idx) in enumerate(expanding_window_split(df=X, start_year=1999, end_year=2020, val_start_year=2015)):
            
            X_train, y_train = X.loc[train_idx], y.loc[train_idx]
            X_val, y_val = X.loc[val_idx], y.loc[val_idx]
            
            default_preprocessor_params = {
                'num_imputer': {}, 
                'num_scaler': True, 
            }
            preprocessor = preprocess_pipeline.build_preprocessor(X, params={'num_imputer': {}, 'num_scaler': True, 'cat': {'cat_imputer': {}}}) # Get Default pipeline

            model_params = {
                'C': 1.0,
                'max_iter': 1000,
                'class_weight': 'balanced'
            }
            model = model_pipeline.get_model_pipeline('logistic_regression', preprocessor, model_params)
            model.fit(X_train, y_train)

            fold_prefix = 'fold_' + str(fold) + '_'
            val_fold_results = evaluate.evaluate_model(model, X_val, y_val, prefix=fold_prefix, verbose=True)
            mlflow.log_metrics(val_fold_results)

            cv_scores.append(val_fold_results[fold_prefix + 'f1'])

        mean_f1 = np.mean(cv_scores)
        mlflow.log_metric('cv_mean_f1', mean_f1)



Initial Dataset Shape: (5025, 85)
Drop Null Target Shape: (4204, 85)
Filter Missingness Shape: (4204, 53)
Filter Low Variance Shape: (4204, 53)
Filter Correlated Shape: (4204, 30)
Final Dataset Shape: (4204, 30)
Baseline Logistic Regression Results
fold_0accuracy: 0.5556
fold_0precision: 0.5551
fold_0recall: 0.5446
fold_0f1: 0.5207

Classification Report
              precision    recall  f1-score   support

           1       0.97      0.63      0.77        52
           2       0.83      0.50      0.62        10
           3       0.47      0.82      0.60        17
           4       0.31      0.50      0.38         8
           5       0.26      0.31      0.29        16
           6       0.37      0.60      0.45        25
           7       0.68      0.44      0.54        43

    accuracy                           0.56       171
   macro avg       0.56      0.54      0.52       171
weighted avg       0.65      0.56      0.57       171


Confusion Matrix
[[33  0  2  2  2  6  7]
 [ 1

KeyboardInterrupt: 

In [27]:
y_train

COUNTRY_PERIOD_INDEX
AFG-1999    7
AFG-2000    7
AFG-2008    7
AFG-2009    7
AFG-2010    7
           ..
ZWE-2010    7
ZWE-2011    7
ZWE-2012    7
ZWE-2013    7
ZWE-2014    7
Name: OECD_RATING, Length: 2628, dtype: object

In [30]:
model['preprocessor']

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x13d1d9850>),
                                Pipeline(steps=[('imputer',
                                                 SimpleImputer(strategy='most_frequent')),
                                                ('encoder',
                                                 OneHotEncoder(handle_unknown='ignore',
                                                               sparse_output=False))])])

In [41]:
model['preprocessor'].transformers[0][1].fit(X, y)

ValueError: Cannot use mean strategy with non-numeric data:
could not convert string to float: 'MEA'

In [31]:
model['preprocessor'].fit(X, y)

ValueError: not enough values to unpack (expected 3, got 2)

## 3.2 - Parameter Tuning Preprocessor

In [68]:
preprocessor_params_grid = {
    # Numeric Transformer
    # Numeric Imputation
    'num_imputer': {
        'knn': {
            'weights': ['uniform', 'distance'],
            'n_neighbors': [3, 5, 7, 11]
        }, 'uni': {
            'strategy': ['mean', 'median', 'most_frequent', 'constant'],
            'fill_value': [0.0]
        }
    },
    # Scalarization
    'num_scaler': [True, False],

    # String Transformer
    'cat': {
        'cat_imputer': {
            'strategy': ['most_frequent', 'constant'],
            'fill_value': ['NaN']
        },
        '': {}
    }
}

In [ ]:
preprocessor = preprocess_pipeline.build_preprocessor(
    X,
    params_grid=preprocessor_params_grid
)

In [ ]:
model_params = {
    'C': 1.0,
    'max_iter': 1000,
    'class_weight': 'balanced'
}

model = model_pipeline.get_model_pipeline(
    model_name,
    preprocessor,
    model_params
)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_preprocessor_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median']
}

search_cv = RandomizedSearchCV(model, param_grid, n_iter=10)

In [ ]:
with mlflow.start_run(run_name='baseline_lr_v1'):

    # Log split metadata
    mlflow.log_params({
        'model': model_name,
        **model_params,
        'train_years': split_cfg['train_years'],
        'val_years': split_cfg['val_years'],
        'test_years': split_cfg['test_years']
    })

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    val_metrics = evaluate.evaluate_model(model, X_val, y_val, prefix='val_')
    test_metrics = evaluate.evaluate_model(model, X_test, y_test, prefix='test_')

    mlflow.log_metrics({**val_metrics, **test_metrics})

    # Log model
    # mlflow.sklearn.log_model(
    #     model,
    #     artifact_path='model',
    #     registered_model_name=None,
    #     input_example=X_test.loc[[X_test.index[0]]]
    # )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

2026/02/08 17:19:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/utils/validat

Baseline Logistic Regression Results
val_accuracy: 0.6839
val_precision: 0.6272
val_recall: 0.6372
val_f1: 0.6255

Classification Report
              precision    recall  f1-score   support

           1       0.93      0.69      0.79       162
           2       0.67      0.58      0.62        24
           3       0.55      0.72      0.62        46
           4       0.46      0.42      0.44        31
           5       0.45      0.60      0.52        40
           6       0.56      0.65      0.60        94
           7       0.76      0.80      0.78       125

    accuracy                           0.68       522
   macro avg       0.63      0.64      0.63       522
weighted avg       0.71      0.68      0.69       522


Confusion Matrix
[[112   1   2   4   1  21  21]
 [  2  14   8   0   0   0   0]
 [  1   6  33   6   0   0   0]
 [  1   0  13  13   4   0   0]
 [  0   0   4   1  24  10   1]
 [  0   0   0   1  23  61   9]
 [  4   0   0   3   1  17 100]]


2026/02/08 17:19:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Baseline Logistic Regression Results
test_accuracy: 0.6090
test_precision: 0.5080
test_recall: 0.4956
test_f1: 0.4931

Classification Report
              precision    recall  f1-score   support

           1       0.78      0.70      0.74       270
           2       0.50      0.33      0.40        33
           3       0.53      0.64      0.58        73
           4       0.12      0.07      0.08        46
           5       0.44      0.36      0.39        84
           6       0.44      0.67      0.53       139
           7       0.74      0.70      0.72       217

    accuracy                           0.61       862
   macro avg       0.51      0.50      0.49       862
weighted avg       0.62      0.61      0.61       862


Confusion Matrix
[[189   1   2   8   0  33  37]
 [ 11  11  11   0   0   0   0]
 [ 13   6  47   2   2   3   0]
 [  6   1  15   3  15   6   0]
 [  4   2   7   8  30  32   1]
 [  6   1   3   3  17  93  16]
 [ 12   0   3   1   4  45 152]]


/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in

MLflow run_id: 19e7ce582e774588a933156a2aadfff2
